<a href="https://colab.research.google.com/github/qhansen628/FMRI_Similarity_CNN/blob/Collecting-updates-2025/Testinf_CNNS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/test.zip

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image_dataset_from_directory

import os
import numpy as np

In [ ]:
def build_classification_dataset(txt_file, batch_size=64, image_size=(32,32), shuffle=True, greyscale=True):
    def parse_line(line):
        parts = tf.strings.split(line)
        img_path = parts[0]
        label = tf.strings.to_number(parts[1], tf.int32)
        return img_path, label

    def load_and_preprocess_image(img_path, label):
        image = tf.io.read_file(img_path)
        image = tf.image.decode_png(image, channels=0)
        image = tf.image.resize(image, image_size)

        # Convert to grayscale
        if greyscale:
          image = tf.image.rgb_to_grayscale(image)

        # Normalize [0,1]
        image = image / 255.0

        return image, label

    ds = tf.data.TextLineDataset(txt_file)
    if shuffle:
        ds = ds.shuffle(10000)

    ds = ds.map(parse_line, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds



# Results from 400 epoch and 100 epoch simple cnn
- 94% on training set for 400 epoch but 22% on test
- 100 epoch got 27% accuracy.
- clearly overfitting and got my hopes up for no reason


In [ ]:
model = load_model('/content/classification_model_100epoch_apr7.keras')

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
model.summary()

Model: "classification_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)             │ (None, 32, 32, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 30, 30, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 15, 15, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 13, 13, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 6, 6, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 2304)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ representation (Dense)               │ (None, 128)                 │         295,040 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 100)                 │          12,900 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 653,514 (2.49 MB)

 Trainable params: 326,756 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 326,758 (1.25 MB)

In [ ]:
batch_size = 512
image_size = (32, 32)

val_ds = build_classification_dataset(
    txt_file='val.txt',
    batch_size=batch_size,
    image_size=image_size,
    shuffle=False,
    greyscale=True
)


In [ ]:
results = model.evaluate(val_ds)
print(f"Validation loss: {results[0]:.4f}")
print(f"Validation accuracy: {results[1]*100:.2f}%")


20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 359ms/step - accuracy: 0.2714 - loss: 4.0036
Validation loss: 3.9999
Validation accuracy: 27.40%


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


# Test CornetZ (alex net style model)

In [ ]:
def lrn_layer(x):
        return tf.nn.local_response_normalization(
            x,
            depth_radius=2,
            bias=1.0,
            alpha=2e-5,
            beta=0.75
        )

cornet_z_model= load_model('/content/cornet_z_model_100epoch_apr8.keras', custom_objects={'lrn_layer': lrn_layer})
cornet_z_model.summary()

Model: "cornet_z_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)             │ (None, 227, 227, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1 (Conv2D)                       │ (None, 111, 111, 64)        │           9,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ norm1 (Lambda)                       │ (None, 111, 111, 64)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ pool1 (MaxPooling2D)                 │ (None, 53, 53, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2 (Conv2D)                       │ (None, 51, 51, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ norm2 (Lambda)                       │ (None, 51, 51, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ pool2 (MaxPooling2D)                 │ (None, 25, 25, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3 (Conv2D)                       │ (None, 23, 23, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ norm3 (Lambda)                       │ (None, 23, 23, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ pool3 (MaxPooling2D)                 │ (None, 11, 11, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv4 (Conv2D)                       │ (None, 9, 9, 512)           │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ norm4 (Lambda)                       │ (None, 9, 9, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ IT (MaxPooling2D)                    │ (None, 4, 4, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 8192)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ fc6 (Dense)                          │ (None, 4096)                │      33,558,528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout6 (Dropout)                   │ (None, 4096)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ fc7 (Dense)                          │ (None, 4096)                │      16,781,312 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout7 (Dropout)                   │ (None, 4096)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ fc8 (Dense)                          │ (None, 100)                 │         409,700 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 156,924,590 (598.62 MB)

 Trainable params: 52,308,196 (199.54 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 104,616,394 (399.08 MB)

In [ ]:
batch_size = 256
image_size = (227,227)

val_ds = build_classification_dataset(
    txt_file='val.txt',
    batch_size=batch_size,
    image_size=image_size,
    shuffle=True,
    greyscale=False
)
results = cornet_z_model.evaluate(val_ds)
print(f"Validation loss: {results[0]:.4f}")
print(f"Validation accuracy: {results[1]*100:.2f}%")

40/40 ━━━━━━━━━━━━━━━━━━━━ 778s 19s/step - accuracy: 0.3880 - loss: 2.7182
Validation loss: 2.7504
Validation accuracy: 38.27%


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
